# EXP-03: uji perbaikan kelas minoritas (ZooLens_final_v2)

Tujuan: membuktikan apakah tambahan data badak dan gajah menaikkan skor kedua kelas itu.

Caranya: model lama (EXP-02) dan model baru (EXP-03) diuji pada **split test yang sama** (259 gambar),
lalu skor **per kelas** dibandingkan. EXP-03 dilatih dengan konfigurasi yang sama seperti EXP-02,
supaya satu-satunya perbedaan adalah datanya.

Sebelum menjalankan: Runtime > Change runtime type > **T4 GPU**.

In [ ]:
!pip -q install ultralytics
from google.colab import drive
drive.mount('/content/drive')

## 1. Atur lokasi file

Ubah tiga path di bawah sesuai lokasi di Google Drive.
- `V2_ZIP`: file `ZooLens_final_v2.zip`
- `EXP02_WEIGHTS`: `best.pt` hasil EXP-02
- `EXP02_ARGS`: `args.yaml` di folder run EXP-02 (boleh dikosongkan, lihat sel konfigurasi)

In [ ]:
V2_ZIP = '/content/drive/MyDrive/ZooLens/ZooLens_final_v2.zip'
EXP02_WEIGHTS = '/content/drive/MyDrive/ZooLens/runs/EXP-02/weights/best.pt'
EXP02_ARGS = '/content/drive/MyDrive/ZooLens/runs/EXP-02/args.yaml'
OUT_DIR = '/content/drive/MyDrive/ZooLens/runs'

import os, zipfile
zipfile.ZipFile(V2_ZIP).extractall('/content')
DATA = '/content/ZooLens_final_v2/data.yaml'
for sp in ['train', 'val', 'test']:
    print(sp, len(os.listdir(f'/content/ZooLens_final_v2/images/{sp}')))

## 2. Skor per kelas model lama (EXP-02) pada split test

Split test v2 sama persis dengan v1, jadi angka ini adalah pembanding yang sah.

In [ ]:
import pandas as pd
from ultralytics import YOLO

def per_class(weights, name):
    m = YOLO(weights).val(data=DATA, split='test', imgsz=640, project='/content/val', name=name, verbose=False)
    b = m.box
    return pd.DataFrame({'kelas': [m.names[i] for i in b.ap_class_index],
                         'precision': b.p, 'recall': b.r, 'mAP50': b.ap50, 'mAP50-95': b.ap}).set_index('kelas').round(3)

base = per_class(EXP02_WEIGHTS, 'exp02_test')
base

## 3. Latih EXP-03 pada data v2

Konfigurasi diambil dari `args.yaml` EXP-02 bila ada. Kalau tidak ada, isi `FALLBACK` dengan
nilai yang dipakai EXP-02 (tanyakan ke Bryan). **Jangan ubah optimizer di eksperimen ini.**

In [ ]:
import yaml
FALLBACK = dict(epochs=50, imgsz=640, batch=16, optimizer='auto', seed=0)
KEYS = ['epochs', 'imgsz', 'batch', 'optimizer', 'lr0', 'lrf', 'momentum', 'weight_decay', 'warmup_epochs',
        'patience', 'seed', 'hsv_h', 'hsv_s', 'hsv_v', 'degrees', 'translate', 'scale', 'shear', 'fliplr',
        'flipud', 'mosaic', 'mixup', 'close_mosaic']
if os.path.exists(EXP02_ARGS):
    old = yaml.safe_load(open(EXP02_ARGS))
    cfg = {k: old[k] for k in KEYS if k in old}
    start = old.get('model', 'yolo11n.pt')
    start = start if start.endswith('.pt') and '/' not in start else 'yolo11n.pt'
    print('Konfigurasi dari EXP-02:', cfg, '| bobot awal:', start)
else:
    cfg, start = FALLBACK, 'yolo11n.pt'
    print('args.yaml EXP-02 tidak ditemukan, memakai FALLBACK:', cfg)

YOLO(start).train(data=DATA, project=OUT_DIR, name='EXP-03', exist_ok=True, **cfg)

## 4. Skor per kelas model baru (EXP-03) pada split test yang sama

In [ ]:
new = per_class(f'{OUT_DIR}/EXP-03/weights/best.pt', 'exp03_test')
new

## 5. Perbandingan dan kesimpulan

Kriteria tuntas:
1. recall dan mAP50 **rhinoceros** dan **elephant** naik,
2. kelas lain tidak turun lebih dari 0,02.

Catatan: test badak hanya 15 gambar, jadi satu gambar bernilai sekitar 0,07 recall.
Kenaikan kecil belum tentu berarti; lihat juga hasil foto uji tim.

In [ ]:
cmp = pd.concat({'EXP-02': base, 'EXP-03': new, 'selisih': (new - base).round(3)}, axis=1)
cmp.to_csv(f'{OUT_DIR}/EXP-03/per_class_test_exp02_vs_exp03.csv')
display(cmp)

d = new - base
minor = ['rhinoceros', 'elephant']; other = [k for k in d.index if k not in minor]
naik = all(d.loc[k, 'recall'] > 0 and d.loc[k, 'mAP50'] > 0 for k in minor)
aman = all(d.loc[k, 'mAP50'] >= -0.02 for k in other)
print('Kelas minoritas naik :', 'YA' if naik else 'TIDAK')
print('Kelas lain tetap aman:', 'YA' if aman else 'TIDAK')
print('KESIMPULAN:', 'kriteria terpenuhi' if naik and aman else 'kriteria belum terpenuhi, lanjut ke oversampling / tambah data')